# Agente com RAG (LangChain) para responder perguntas sobre um documento

**Autora:** Laura Silva Soares de Melo — projeto final de Tópicos Contemporâneos, pós-graduação em Engenharia e Análise de Dados (CESAR School)

**Documento usado:** slide de aula "Agents" (PDF de 21 páginas, material da disciplina, **não incluído** neste repositório; use qualquer PDF curto)

---

## Objetivo

Este notebook implementa um sistema RAG (Retrieval-Augmented Generation) com **LangChain** e um **Agent** capaz de decidir quando consultar o documento indexado para responder perguntas. As etapas são:

1. Escolha do Documento
2. Splitting do Documento
3. Criação do Vector Store (FAISS + embeddings)
4. Retrieval (indexação + prompt do agent)
5. Geração de Respostas e **avaliação crítica dos resultados**


## 1. Escolha do Documento

**Documento:** slide de aula **"Agents"** da própria disciplina (21 páginas), que cobre: definição de agentes, frameworks de pensamento (Chain-of-Thought, Tree-of-Thoughts, Graph-of-Thoughts), engenharia de prompts, sistemas multi-agente, loops de agente (Reactive, Plan-Act-Observe, ReAct, Reflection, Multi-Agent Collaborative), desafios e limitações, dicas de prompt e o **GCIF Prompt Framework** — inclusive um exemplo específico de prompt para *RAG com Agents*.

**Justificativa da escolha:**

- É o **próprio material da disciplina sobre o tema do projeto** (Agents + RAG). Isso é uma vantagem direta para a etapa de avaliação: a equipe conhece o conteúdo de cor e consegue julgar com precisão se as respostas do agent estão corretas, incompletas ou incorretas, comparando com o que foi apresentado em aula.
- É um documento **compacto** (21 páginas, ~5.900 caracteres de texto extraível), o que torna viável revisar manualmente cada chunk gerado e cada resposta do agent — importante para um projeto de avaliação qualitativa.
- É um documento **realista e imperfeito**: como é um export de slide, algumas páginas têm o conteúdo principal embutido como **imagem** (ex: o diagrama de "Frameworks de pensamento" na página 8 e o print do prompt GCIF-RAG na página 18), em vez de texto selecionável. Isso serve como um teste real e interessante dos limites de um pipeline de RAG baseado puramente em extração de texto — e é discutido como um problema encontrado na seção de avaliação (seção 5.1).
- Por ser um arquivo próprio do grupo (não uma URL pública), o notebook fica independente de conexão com sites externos: o próprio grupo faz o upload do PDF ao rodar o notebook no Colab.

O carregamento é feito por upload interativo do arquivo `Agents.pdf` diretamente no Google Colab (célula abaixo).


In [ ]:
#Bibliotecas
import os
from getpass import getpass

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.tools.retriever import create_retriever_tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq


In [ ]:
# Upload do documento (arquivo próprio do grupo — o slide "Agents.pdf" da disciplina)

PDF_PATH = "Agents.pdf"

print(f"Arquivo selecionado: {PDF_PATH}")



In [ ]:
# Validaçao da existência do arquivo PDF

import os

PDF_PATH = "Agents.pdf"

print(os.path.exists(PDF_PATH))

In [ ]:
# Carregamento do documento com LangChain
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

total_chars = sum(len(d.page_content) for d in docs)
print(f"Número de páginas carregadas: {len(docs)}")
print(f"Total de caracteres extraídos: {total_chars}")
print(f"Média de caracteres por página: {total_chars/len(docs):.0f}")

print("\n--- Páginas com pouco texto extraído (provável conteúdo em imagem/diagrama) ---")
for i, d in enumerate(docs):
    if len(d.page_content.strip()) < 40:
        print(f"  página {i}: apenas {len(d.page_content)} chars -> {d.page_content!r}")

print("\n--- Amostra da página 7 (Definição de Agent) ---\n")
print(docs[7].page_content[:400])


Número de páginas carregadas: 21
Total de caracteres extraídos: 5889
Média de caracteres por página: 280

--- Páginas com pouco texto extraído (provável conteúdo em imagem/diagrama) ---
  página 1: apenas 23 chars -> 'TÓPICOS \nCONTEMPORÂNEOS'
  página 6: apenas 6 chars -> 'Agents'
  página 8: apenas 24 chars -> 'Frameworks de pensamento'
  página 12: apenas 12 chars -> 'Multi-agents'

--- Amostra da página 7 (Definição de Agent) ---

Deﬁnição
● Conceito: 
○ Um sistema capaz de perceber seu ambiente, 
raciocinar e tomar ações para atingir um 
objetivo especíﬁco.
● Diferença Chave:
○ Chatbot: Responde perguntas (Baseado em 
texto).
○  Agente: Executa tarefas (Baseado em ação).
● Analogia:
○  O LLM é o "cérebro", o Agente é o "corpo" com 
mãos e ferramentas.


## 2. Splitting do Documento

Utilizamos o `RecursiveCharacterTextSplitter` do LangChain, que tenta quebrar o texto respeitando primeiro parágrafos (`\n\n`), depois linhas (`\n`), depois marcadores de bullet (`● `) e frases (`. `), e só em último caso quebra no meio de palavras. Isso preserva melhor a estrutura de tópicos/bullets típica de um slide.

**Particularidade deste documento:** por ser um slide de aula, a maioria das páginas já é curta (a maior parte tem entre 150 e 450 caracteres — bem menos do que uma página de um artigo científico, por exemplo). Isso significa que, na prática, boa parte das páginas vira um único chunk, independentemente do `chunk_size` escolhido — a diferença entre configurações aparece principalmente nas poucas páginas mais longas, como a página 4 (regras de nota do projeto final, ~900 caracteres).

**Testes realizados:** comparamos três combinações de `chunk_size` / `chunk_overlap`, usando valores menores do que seriam usados em um documento denso (ex: um paper), justamente por causa do tamanho médio pequeno das páginas deste slide.


In [ ]:
# Teste de diferentes configurações de splitting
configs = [
    {"chunk_size": 200, "chunk_overlap": 30},
    {"chunk_size": 400, "chunk_overlap": 50},
    {"chunk_size": 600, "chunk_overlap": 80},
]

resultados_splitting = []
for cfg in configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"],
        chunk_overlap=cfg["chunk_overlap"],
        separators=["\n\n", "\n", "\u25cf ", ". ", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    lens = [len(c.page_content) for c in chunks]
    resultados_splitting.append({
        **cfg,
        "n_chunks": len(chunks),
        "avg_len": sum(lens) / len(lens),
        "min_len": min(lens),
        "max_len": max(lens),
    })

for r in resultados_splitting:
    print(f"chunk_size={r['chunk_size']:<5} overlap={r['chunk_overlap']:<4} -> "
          f"{r['n_chunks']:>3} chunks | tamanho médio={r['avg_len']:.0f} "
          f"(min={r['min_len']}, max={r['max_len']})")


chunk_size=200   overlap=30   ->  45 chunks | tamanho médio=133 (min=6, max=200)
chunk_size=400   overlap=50   ->  28 chunks | tamanho médio=216 (min=6, max=400)
chunk_size=600   overlap=80   ->  22 chunks | tamanho médio=270 (min=6, max=557)


Configuração final escolhida: chunk_size=400, chunk_overlap=50


-  Justificativa: como a maioria das páginas do slide já é curta (~280 chars em média), um chunk_size de 400 é grande o suficiente para manter cada slide como um único chunk na maioria dos casos (preservando o tópico completo), mas pequeno o suficiente para dividir as poucas páginas mais longas (como a página 4, com as regras de avaliação) em partes menores e mais específicas, evitando misturar assuntos distintos dentro de um mesmo chunk. O overlap de 50 caracteres (~12%) evita cortar uma ideia bem no meio quando uma página precisa ser dividida.

In [ ]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", "\u25cf ", ". ", " ", ""],
)

chunks = text_splitter.split_documents(docs)
print(f"Total de chunks gerados: {len(chunks)}")
print("\n--- Exemplo de chunk (definição de Agent) ---\n")
print(chunks[7].page_content if len(chunks) > 7 else chunks[-1].page_content)


Total de chunks gerados: 28

--- Exemplo de chunk (definição de Agent) ---

mais documentos de sua escolha. Podendo ser arquivos PDF, de texto, páginas da web etc. O projeto será avaliado nos 
seguintes aspectos:
● Escolha do Documento
● Splitting do Documento
● Criação de Vector Store
● Retrieval
● Geração de Respostas


## 3. Criação do Vector Store

Para gerar os embeddings, usamos o modelo `sentence-transformers/all-MiniLM-L6-v2` via `HuggingFaceEmbeddings`. Escolhemos esse modelo por ser:

- **Gratuito e local** — não exige API key nem custo por chamada, roda direto no ambiente do Colab (CPU já é suficiente).
- **Leve e rápido** — dimensão de embedding relativamente pequena (384), adequado para o volume pequeno de chunks deste documento (poucas dezenas).
- Apesar de o modelo ser treinado majoritariamente em inglês, ele também apresenta desempenho razoável em português por ter sido exposto a corpora multilíngues — o que é suficiente para um documento curto como este, mas é discutido como ponto de atenção na avaliação (seção 5.1).

Como vector store, usamos o **FAISS** (Facebook AI Similarity Search), por ser simples de usar localmente dentro do Colab, sem precisar de um serviço externo (ex: Pinecone, Chroma Cloud etc.), o que facilita a reprodutibilidade do notebook.


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"Índice FAISS criado com {vectorstore.index.ntotal} vetores.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5969.43it/s]


Índice FAISS criado com 28 vetores.


## 4. Retrieval

### 4.1 Testes dos parâmetros de indexação (k)

Testamos diferentes valores de `k` (número de chunks retornados pela busca) para uma mesma pergunta, avaliando manualmente se os trechos recuperados são de fato relevantes. Como o documento é pequeno (poucas dezenas de chunks), testamos valores de k proporcionalmente menores do que usaríamos em um documento grande.


In [ ]:
query_teste = "Quais são os tipos de loop de um agente?"

for k in [1, 2, 4]:
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    resultados = retriever.invoke(query_teste)
    print(f"\n===== k={k} =====")
    for i, r in enumerate(resultados, 1):
        print(f"[{i}] (página {r.metadata.get('page', '?')}) {r.page_content[:150].strip()}...")



===== k=1 =====
[1] (página 14) Desaﬁos e Limitações
● Loops Inﬁnitos: Quando o agente ﬁca preso em uma 
tarefa sem saída.
● Alucinações de Ação: Tentar usar ferramentas que 
não exi...

===== k=2 =====
[1] (página 14) Desaﬁos e Limitações
● Loops Inﬁnitos: Quando o agente ﬁca preso em uma 
tarefa sem saída.
● Alucinações de Ação: Tentar usar ferramentas que 
não exi...
[2] (página 7) Deﬁnição
● Conceito: 
○ Um sistema capaz de perceber seu ambiente, 
raciocinar e tomar ações para atingir um 
objetivo especíﬁco.
● Diferença Chave:
○...

===== k=4 =====
[1] (página 14) Desaﬁos e Limitações
● Loops Inﬁnitos: Quando o agente ﬁca preso em uma 
tarefa sem saída.
● Alucinações de Ação: Tentar usar ferramentas que 
não exi...
[2] (página 7) Deﬁnição
● Conceito: 
○ Um sistema capaz de perceber seu ambiente, 
raciocinar e tomar ações para atingir um 
objetivo especíﬁco.
● Diferença Chave:
○...
[3] (página 9) incluindo uma deﬁnição técnica e um exemplo prático....
[4] (página 5) Fiz o trabalho 

**Observação sobre os testes de k:** como muitas páginas viram um único chunk pequeno, um `k` muito baixo (1) pode não trazer todo o contexto de um tópico espalhado por mais de uma página. Em contrapartida, um `k` alto para um documento tão pequeno passa a incluir chunks de assuntos diferentes. Na pergunta de teste ("tipos de loop de um agente"), **nenhum dos valores testados recuperou a página 13**, que trata exatamente desse tema; esse é um problema de retrieval que aparece de novo na avaliação (seção 5.1). O valor final usado foi **k=2**, também para respeitar o limite de tokens por minuto do tier gratuito da Groq.

Configuração final do retriever k reduzido para 2 (em vez de 3) para manter cada chamada ao LLM dentro do limite de tokens por minuto do tier gratuito da Groq.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

### 4.2 Construção do Agent e Prompt

Diferente de uma simples *chain* de RAG (que sempre busca no documento antes de responder), aqui construímos um **Agent**: o modelo decide, a cada pergunta, se precisa ou não consultar a ferramenta de busca no documento (`busca_no_documento`) antes de responder. Isso é feito com `create_tool_calling_agent` + `AgentExecutor` do LangChain.

**Prompt do agent**, construído seguindo o próprio **GCIF Prompt Framework** (Objetivo, Contexto, Instruções, Formato) apresentado no material da disciplina, adaptado para um agent com tool-calling (em vez de injeção manual do contexto, o próprio agent decide quando chamar a ferramenta de busca):


Objetivo:
Responder às perguntas do usuário utilizando exclusivamente o conteúdo do
documento indexado (slide "Agents" da disciplina).

Contexto:
Você é um agente especializado no material de aula sobre Agentes de IA
(definição, frameworks de pensamento, prompts, loops de agente, desafios e
RAG). Você tem acesso à ferramenta 'busca_no_documento', que retorna trechos
relevantes do documento indexado.

Instruções:
- Sempre utilize a ferramenta 'busca_no_documento' antes de responder
  perguntas sobre o conteúdo do material.
- Use apenas as informações retornadas pela ferramenta. Não utilize
  conhecimento externo.
- Caso a informação não esteja nos trechos recuperados, responda:
  "Não encontrei essa informação na base disponível."
- Sempre indique a página do documento de onde veio a informação usada.
- Seja claro e direto.

Formato de saída:
- Responda em tópicos curtos.
- Inclua ao final uma seção chamada "Fontes" com as páginas usadas.
- Responda em português.


In [ ]:
retriever_tool = create_retriever_tool(
    retriever,
    name="busca_no_documento",
    description=(
        "Busca trechos relevantes do slide 'Agents' da disciplina (definição de "
        "agentes, frameworks de pensamento, prompts, loops de agente, desafios e "
        "RAG). Use esta ferramenta sempre que a pergunta do usuário for sobre o "
        "conteúdo do material."
    ),
)

SYSTEM_PROMPT = '''Objetivo:
Responder às perguntas do usuário utilizando exclusivamente o conteúdo do
documento indexado (slide "Agents" da disciplina).

Contexto:
Você é um agente especializado no material de aula sobre Agentes de IA
(definição, frameworks de pensamento, prompts, loops de agente, desafios e
RAG). Você tem acesso à ferramenta 'busca_no_documento', que retorna trechos
relevantes do documento indexado.

Instruções:
- Sempre utilize a ferramenta 'busca_no_documento' antes de responder
  perguntas sobre o conteúdo do material.
- Use apenas as informações retornadas pela ferramenta. Não utilize
  conhecimento externo.
- Caso a informação não esteja nos trechos recuperados, responda:
  "Não encontrei essa informação na base disponível."
- Sempre indique a página do documento de onde veio a informação usada.
- Seja claro e direto.

Formato de saída:
- Responda em tópicos curtos.
- Inclua ao final uma seção chamada "Fontes" com as páginas usadas.
- Responda em português.'''

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])


**Escolha do LLM:** usamos o **Groq** (`langchain-groq`) com o modelo `llama-3.1-8b-instant`, pois oferece uma **API gratuita** (com limites de uso generosos para fins acadêmicos) e respostas rápidas, o que evita custos para o grupo rodar e testar o notebook várias vezes.

> Para o funcinamento do Código crie uma chave gratuita em https://console.groq.com/keys e cole abaixo quando solicitado. Se preferir usar a OpenAI, veja o comentário alternativo na célula seguinte.


In [ ]:

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Cole sua GROQ_API_KEY (gratuita em console.groq.com/keys): ")

# max_tokens limita o tamanho da resposta gerada, e max_iterations limita
# quantas vezes o agent pode chamar a ferramenta de busca dentro de uma
# mesma pergunta. Sem esses limites, o tier gratuito da Groq (6.000 tokens
# por minuto para este modelo) pode estourar com "APIStatusError: 413 -
# Request too large" em perguntas que levam o agent a buscar várias vezes.


llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_tokens=512)

# --- Alternativa com OpenAI (descomente e ajuste se preferir) ---
# from langchain_openai import ChatOpenAI
# if "OPENAI_API_KEY" not in os.environ:
#     os.environ["OPENAI_API_KEY"] = getpass("Cole sua OPENAI_API_KEY: ")
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_tool_calling_agent(llm, [retriever_tool], prompt)
agent_executor = AgentExecutor(
    agent=agent, tools=[retriever_tool], verbose=True, max_iterations=3
)

print("Agent RAG pronto.")


Agent RAG pronto.


## 5. Geração de Respostas

Testamos o agent com um conjunto de perguntas: algumas claramente respondíveis a partir do texto extraído do documento, uma pergunta sobre um trecho que sabemos estar embutido como **imagem** (página 18 — para testar o limite do pipeline), e uma pergunta totalmente fora do escopo do documento, para verificar se o agent reconhece corretamente os limites do que sabe (evitando alucinação).


In [ ]:
import time
from groq import APIStatusError

perguntas_teste = [
    "Qual é a definição de Agent e qual a diferença chave entre um Agent e um Chatbot?",
    "Quais são os tipos de loop de agente apresentados no material?",
    "Quais são os desafios e limitações de agentes citados no material?",
    "O que é o GCIF Prompt Framework e quais são seus quatro componentes?",
    "Qual é exatamente o prompt sugerido no material para um agente RAG usando o framework GCIF?",  # conteúdo está em imagem (página 18) — teste de limite
    "Qual é a capital da França?",  # pergunta fora do escopo do documento, de propósito
]

respostas = []
for pergunta in perguntas_teste:
    print(f"\n{'='*80}\nPERGUNTA: {pergunta}\n{'='*80}")
    try:
        resultado = agent_executor.invoke({"input": pergunta})
        resposta = resultado["output"]
    except APIStatusError as e:
        # Limite de tokens/minuto do tier gratuito da Groq estourado nesta
        # pergunta específica — aguarda um pouco e tenta mais uma vez.
        print(f"Limite de taxa da Groq atingido, aguardando 20s e tentando de novo... ({e})")
        time.sleep(20)
        resultado = agent_executor.invoke({"input": pergunta})
        resposta = resultado["output"]
    print(f"\nRESPOSTA:\n{resposta}")
    respostas.append({"pergunta": pergunta, "resposta": resposta})
    time.sleep(3)  # pequena pausa entre perguntas para não estourar o limite por minuto



PERGUNTA: Qual é a definição de Agent e qual a diferença chave entre um Agent e um Chatbot?


> Entering new AgentExecutor chain...

Invoking: `busca_no_documento` with `{'query': 'definição de Agent e diferença chave entre um Agent e um Chatbot'}`


Deﬁnição
● Conceito: 
○ Um sistema capaz de perceber seu ambiente, 
raciocinar e tomar ações para atingir um 
objetivo especíﬁco.
● Diferença Chave:
○ Chatbot: Responde perguntas (Baseado em 
texto).
○  Agente: Executa tarefas (Baseado em ação).
● Analogia:
○  O LLM é o "cérebro", o Agente é o "corpo" com 
mãos e ferramentas.

Agents"Não encontrei a diferença chave entre um Agent e um Chatbot na base disponível."

Fontes:
Página 3 do documento "Agents" da disciplina.

> Finished chain.

RESPOSTA:
"Não encontrei a diferença chave entre um Agent e um Chatbot na base disponível."

Fontes:
Página 3 do documento "Agents" da disciplina.

PERGUNTA: Quais são os tipos de loop de agente apresentados no material?


> Entering new AgentExecutor chai

### 5.1 Avaliação da equipe

**Qualidade geral das respostas:**
- [x] As respostas às perguntas 1-4 (conteúdo em texto puro) foram corretas e batem com o que foi apresentado em aula? **Não, na maioria dos casos.** Em 3 das 4 perguntas (1, 3 e 4), o retriever recuperou o trecho **certo** do documento (a definição de Agent na pág. 7, os desafios/limitações na pág. 14, e o GCIF Prompt Framework na pág. 17 — com seus quatro componentes: Objetivo, Contexto, Instruções e Formato de saída), mas o agent respondeu **"Não encontrei essa informação na base disponível"** mesmo com a resposta certa já presente no contexto retornado pela ferramenta. Ou seja: o retrieval funcionou, mas a geração falhou em usar o contexto corretamente (falso negativo). Apenas a pergunta 3 teve uma resposta parcialmente correta (mencionou "alucinações de ação" antes de negar).
- [x] O agent citou corretamente a página de onde veio a informação, como pedido no prompt (formato "Fontes")? **Não.** Nas perguntas 1 e 3 o agent citou "Página 3" como fonte, mas o trecho realmente usado veio das páginas 7 e 14, respectivamente. Isso não é o modelo "mentindo" por má-fé: a ferramenta `busca_no_documento` (criada com `create_retriever_tool`) retorna só o **texto** dos chunks recuperados, sem incluir o número da página na string que é passada pro LLM — então o modelo não tem como saber a página real e acaba **inventando um número**. Esse é um problema concreto e corrigível (dá pra resolver passando um `document_prompt` customizado pro `create_retriever_tool`, incluindo a página junto de cada trecho).
- [x] Na pergunta 5 (sobre o prompt GCIF-RAG da página 18, que está embutido como **imagem** no PDF), o agent respondeu corretamente que não encontrou a informação, ou tentou "adivinhar"/alucinar uma resposta? **Respondeu corretamente** — "Não encontrei a informação específica sobre o prompt sugerido para um agente RAG usando o framework GCIF na base disponível." Isso confirma a limitação identificada na seção 1: o conteúdo da página 18 é uma imagem e não foi indexado como texto, então o agent genuinamente não tem como responder — e, nesse caso (ao contrário das perguntas 1, 3 e 4), o "não encontrei" está correto.
- [x] Na pergunta 6 (fora do escopo do documento), o agent reconheceu que a informação não está no documento, em vez de responder com conhecimento geral? **Sim, mas de forma ineficiente.** O agent não alucinou uma resposta sobre a capital da França, mas ficou preso em um loop, chamando a ferramenta de busca repetidamente (mais de 10 vezes), cada vez reformulando a query adicionando mais palavras à pergunta anterior, até finalmente desistir e responder "Não encontrei". No fim ainda listou como "Fontes" as páginas 1 a 10 do documento — uma lista genérica que não reflete de fato quais trechos foram usados (outra citação pouco confiável).

**Pontos fortes observados:**
- O retrieval em si (FAISS + embeddings) funcionou bem na maioria dos casos: para 3 das 4 perguntas de conteúdo, o chunk certo apareceu entre os resultados retornados pela ferramenta.
- O agent nunca "inventou" conteúdo que não estava no documento — mesmo errando ao dizer "não encontrei", ele não alucinou fatos externos ao material (o comportamento de segurança do prompt funcionou nesse sentido).
- O caso de teste da página-imagem (pergunta 5) e o caso fora do escopo (pergunta 6) tiveram o resultado esperado: o agent reconheceu os limites do que sabia.

**Pontos de melhoria / problemas encontrados:**
- **Falso negativo na geração (problema principal):** o modelo `llama-3.1-8b-instant` (gratuito, via Groq) recebeu o contexto correto em 3 perguntas e mesmo assim respondeu que não encontrou a informação. Isso indica que o modelo é conservador demais ao seguir a instrução "responda apenas com base no CONTEXTO" — provavelmente por ser um modelo pequeno (8B parâmetros), ele tem mais dificuldade em extrair a resposta certa de um contexto com múltiplos trechos concatenados do que um modelo maior.
- **Citação de página não confiável:** como descrito acima, a ferramenta de busca não repassa o número da página pro LLM, então as citações em "Fontes" são inventadas, não reais.
- **Falha de retrieval na pergunta 2:** para "quais são os tipos de loop de agente", o retriever nunca trouxe a página 13 (que é exatamente a seção "Loop" do slide, com Simple Reactive Loop, Plan-Act-Observe, ReAct, Reflection, Multi-Agent Collaborative) — nem mesmo testando k=1, 2 ou 4 na seção 4.1. O chunk da página 14 ("Desafios e Limitações", que só cita "Loops Infinitos" de passagem) teve prioridade na busca por similaridade, provavelmente porque a palavra "loop" aparece lá também, competindo com a página certa.
- **Loop excessivo de tool-calling na pergunta fora do escopo:** o agent chamou a ferramenta de busca mais de 10 vezes para a pergunta sobre a capital da França, reformulando a query a cada tentativa, antes de desistir — um comportamento ineficiente (gasta tempo e tokens à toa) mesmo chegando à resposta certa no final.
- **Tempo/custo:** por causa desse loop, essa pergunta específica consumiu bem mais tokens e tempo do que as outras, o que é relevante considerando o limite de tokens/minuto do tier gratuito da Groq.

**Possíveis melhorias futuras:**
- Corrigir a citação de fontes: passar a página junto do texto de cada chunk retornado pela ferramenta (via `document_prompt` no `create_retriever_tool`), em vez de deixar o modelo adivinhar.
- Testar um LLM maior/mais capaz (ex: `llama-3.1-70b` ou GPT-4o-mini) para ver se o problema de falso negativo (não usar um contexto correto) melhora — é provável que seja uma limitação do tamanho do modelo, não da arquitetura do pipeline.
- Reforçar no prompt do agent uma instrução explícita tipo "leia atentamente todos os trechos retornados antes de decidir se a resposta não está lá", já que o modelo está sendo conservador demais.
- Aplicar OCR (ex: com `pytesseract` ou um loader multimodal) nas páginas com conteúdo em imagem, para recuperar também esse conteúdo (páginas 8 e 18).
- Testar um modelo de embeddings multilíngue/otimizado para português, já que as perguntas são em português mas o modelo usado é predominantemente treinado em inglês — pode ajudar a resolver a falha de retrieval da pergunta 2.
- Limitar melhor o número de reformulações de query que o agent tenta antes de desistir, para evitar o loop longo observado na pergunta 6.


## Conclusão

Este notebook implementou um pipeline completo de RAG com Agent usando LangChain sobre o material da própria disciplina: carregamento do documento via upload no Colab, splitting configurável, criação de um vector store com FAISS e um agent capaz de decidir quando consultar o documento para responder, com prompt construído a partir do GCIF Prompt Framework ensinado em aula. As escolhas de parâmetros (chunk_size, overlap, k, modelo de embeddings e LLM) foram testadas e justificadas ao longo do notebook, e a qualidade das respostas — incluindo uma limitação real do documento (conteúdo em imagem) — foi avaliada qualitativamente pela equipe na seção 5.1.
